# Urdu Question Generation

Run the cells in order to prepare data, train the tokenizer, and train the model.

In [ ]:
%pip install -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.5 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import torch

# dirs
ROOT_DIR = Path(__file__).parent

DATA_DIR = ROOT_DIR / "data"
TRAIN_DATA = DATA_DIR / "train.tsv"
VALID_DATA = DATA_DIR / "valid.tsv"

ARTIFACTS_DIR = ROOT_DIR / "artifacts"
TOKENIZER_DIR = ARTIFACTS_DIR / "tokenizer"
MODEL_PREFIX = "ur_sp"
TOKENIZER_MODEL = TOKENIZER_DIR / f"{MODEL_PREFIX}.model"
CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints"
BEST_MODEL = CHECKPOINT_DIR / "best_model.pt"

RESULTS_DIR = ROOT_DIR / "results"
FIGS_DIR = RESULTS_DIR / "figures"

# data & prep
DATASET_NAME = "uqa/UQA"

MAX_SOURCE_LENGTH = 60
MAX_TARGET_LENGTH = 25

ANS_OPEN = "<ans>"
ANS_CLOSE = "</ans>"
SENT_DELIMS = "\u06D4\u061F!"

# model
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.3
TEACHER_FORCING_RATIO = 0.5

# training
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 10

# tokenizer
VOCAB_SIZE = 8000

PAD_IDX = 0
UNK_IDX = 1
SOS_IDX = 2
EOS_IDX = 3


# device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ModuleNotFoundError: No module named 'scripts'

In [ ]:
from config import *
from datasets import load_dataset
import csv
import matplotlib.pyplot as plt

In [ ]:
def split_sentences(text):
    """take the paragraph and extract complete sentences from it

    Args:
        text (str): context paragraph

    Yields:
        tuple: a tuple containing the start index, end index, and the extracted sentence
    """
    start = 0
    for i, ch in enumerate(text):

        if ch in SENT_DELIMS:
            yield start, i + 1, text[start : i + 1]
            start = i + 1

    if start < len(text):
        yield start, len(text), text[start:]

In [ ]:
def make_pair(example, max_src=MAX_SOURCE_LENGTH, max_tgt=MAX_TARGET_LENGTH):
    """make pairs that will be used for training
    src is a sentence with <ans>...</ans> marked
    tgt is the question entry of example

    Args:
        example (dict): a single example from the dataset
        max_src (int, optional): maximum length for the source sequence, defaults to MAX_SOURCE_LENGTH
        max_tgt (int, optional): maximum length for the target sequence, defaults to MAX_TARGET_LENGTH

    Returns:
        tuple: a tuple containing the source and target sequences, or None if the example is not suitable for training
    """
    if example["is_impossible"]:
        return None
    
    a_start = example["answer_start"]
    a_text = example["answer"]
    context = example["context"]

    for s, e, sent in split_sentences(context):
        if s <= a_start < e:

            rel = a_start - s
            if sent[rel : rel + len(a_text)] != a_text:
                return None
            
            src = (sent[:rel] + " " + ANS_OPEN + " " + a_text + " " + ANS_CLOSE + " " + sent[rel + len(a_text) :]).strip()
            src = " ".join(src.split())
            tgt = " ".join(example["question"].split())

            if len(src.split()) > max_src or len(tgt.split()) > max_tgt:
                return None
            
            return src, tgt
        
    return None

README.md:   0%|          | 0.00/898 [00:00<?, ?B/s]

data/train-00000-of-00001-bac007e8ca7192(…): reconstructing file:   0%|          |  0.00B / 30.2MB            

data/train-00000-of-00001-bac007e8ca7192(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-cf8a6960d(…): reconstructing file:   0%|          |  0.00B / 2.92MB            

data/validation-00000-of-00001-cf8a6960d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/124745 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/16824 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 124745
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 16824
    })
})


In [ ]:
def build_split(split, out_path):
    """make pairs of the split(train/valid) and store it

    Args:
        split (Dataset): split passed that need to be processed and stored
        out_path (Path): output path where the processed pairs will be stored
    """
    pairs = [p for p in map(make_pair, split) if p is not None]

    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\")
        w.writerows(pairs)
    print(f"{out_path}: {len(pairs)} pairs")

    return pairs

<built-in method keys of dict object at 0x7d75ad6a1b80>


In [ ]:
ds = load_dataset(DATASET_NAME)
print(ds)

ex = ds["train"][0]
print(ex.keys())
print(ex["question"])
print(ex["answer"])

n_total = len(ds["train"])
n_ans = n_total - sum(ds["train"]["is_impossible"])
print (f"train rows: {n_total}, answerable: {n_ans}")

DATA_DIR.mkdir(parents=True, exist_ok=True)
train_pairs = build_split(ds["train"], TRAIN_DATA)
valid_pairs = build_split(ds["validation"], VALID_DATA)

بیونس نے کب مقبولیت حاصل کرنا شروع کی؟
1990 کی دہائی کے آخر میں


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / "data_counts.txt", "w", encoding="utf-8") as f:
    f.write(f"train: \n\ttotal: {len(ds['train'])} \n\tanswerable: {len(ds['train']) - sum(ds['train']['is_impossible'])} \n\tselected: {len(train_pairs)}\n")
    f.write(f"validation: \n\ttotal: {len(ds['validation'])} \n\tanswerable: {len(ds['validation']) - sum(ds['validation']['is_impossible'])} \n\tselected: {len(valid_pairs)}\n")

FIGS_DIR.mkdir(parents=True, exist_ok=True)
plt.figure()
plt.hist([len(src.split()) for src, _ in train_pairs], bins=30, label="train source")
plt.title("distribution of source lengths in train")
plt.xlabel("number of words")
plt.ylabel("count")
plt.savefig(FIGS_DIR / "train_source_lengths.png")

plt.figure()
plt.hist([len(tgt.split()) for _, tgt in train_pairs], bins=30, label="train target")
plt.title("distribution of target lengths in train")
plt.xlabel("number of words")
plt.ylabel("count")
plt.savefig(FIGS_DIR / "train_target_lengths.png")

train rows: 124745, answerable: 83018


In [ ]:
plt.figure()
plt.hist([len(src.split()) for src, _ in valid_pairs], bins=30, label="valid source")
plt.title("distribution of source lengths in valid")
plt.xlabel("number of words")
plt.ylabel("count")
plt.savefig(FIGS_DIR / "valid_source_lengths.png")

plt.figure()
plt.hist([len(tgt.split()) for _, tgt in valid_pairs], bins=30, label="valid target")
plt.title("distribution of target lengths in valid")
plt.xlabel("number of words")
plt.ylabel("count")
plt.savefig(FIGS_DIR / "valid_target_lengths.png")

In [ ]:
from config import *
import sentencepiece as spm
import csv


def read_split(in_path):
    """read the data and split it into source and target sequences

    Args:
        in_path (str): path to the input file

    Returns:
        list: a list of tuples containing the source and target sequences
    """
    pairs = []

    with open(in_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\")
        for row in reader:
            src, tgt = row
            pairs.append((src, tgt))

    return pairs

In [ ]:
train_pairs = read_split(TRAIN_DATA)

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
with open(TOKENIZER_DIR / "sp_corpus.txt", "w", encoding="utf-8") as f:
    for src, tgt in train_pairs:
        f.write(src + "\n" + tgt + "\n")

spm.SentencePieceTrainer.train(
    input=str(TOKENIZER_DIR / "sp_corpus.txt"),
    model_prefix=str(TOKENIZER_DIR / MODEL_PREFIX),
    vocab_size=VOCAB_SIZE,
    model_type="unigram",
    character_coverage=1.0,
    user_defined_symbols=[ANS_OPEN, ANS_CLOSE],
    pad_id=PAD_IDX,
    unk_id=UNK_IDX,
    bos_id=SOS_IDX,
    eos_id=EOS_IDX,
)

In [ ]:
sp = spm.SentencePieceProcessor(model_file=str(TOKENIZER_MODEL))
src, tgt = train_pairs[0]
print(sp.encode(src, out_type=str))
print(sp.encode(tgt))
print(sp.decode(sp.encode(tgt)) == tgt)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / "5_tokenizer_results.txt", "w", encoding="utf-8") as f:
    for src, _ in train_pairs[:5]:
        f.write(f"original: {src}\ntokenized: {sp.encode(src, out_type=str)}\n\n")

In [ ]:
from config import *
import sentencepiece as spm
from scripts.train_tokenizer import read_split
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from app.model import Seq2Seq
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
class UQADataset(Dataset):

    def __init__(self, pairs):
        """initialize the dataset with pairs of source and target sequences

        Args:
            pairs (list): a list of tuples containing source and target sequences
        """
        self.pairs = pairs

    def __len__(self):
        """get the length of the dataset

        Returns:
            int: the number of pairs in the dataset
        """
        return len(self.pairs)

    def __getitem__(self, idx):
        """get a specific pair from the dataset

        Args:
            idx (int): the index of the pair to retrieve

        Returns:
            tuple: a tuple containing the source and target sequences
        """
        return self.pairs[idx]

In [ ]:
def collate_fn(batch, pad_idx):
    """collate a batch of sequences

    Args:
        batch (list): a list of tuples containing source and target sequences
        pad_idx (int): the index to use for padding

    Returns:
        tuple: a tuple containing the padded source and target sequences
    """
    src_batch, tgt_batch = zip(*batch)

    src_batch = [torch.tensor(src) for src in src_batch]
    tgt_batch = [torch.tensor(tgt) for tgt in tgt_batch]

    src_batch = pad_sequence(src_batch, padding_value=pad_idx)
    tgt_batch = pad_sequence(tgt_batch, padding_value=pad_idx)

    return src_batch, tgt_batch

In [ ]:
def train_step(dataloader, model, optimizer, criterion, device, pad_idx):
    """perform a training on one epoch

    Args:
        dataloader (DataLoader): the data loader for the training data
        model (Seq2Seq): the sequence-to-sequence model
        optimizer (torch.optim.Optimizer): the optimizer to use for training
        criterion (torch.nn.Module): the loss function to use
        device (torch.device): the device to use for training
        pad_idx (int): the index to use for padding

    Returns:
        float: the average loss for the epoch
    """
    model.train()
    total_loss = 0

    tqdm_bar = tqdm(dataloader, desc="Training", leave=False)
    for idx, (src, tgt) in enumerate(tqdm_bar):
        src, tgt = src.to(device), tgt.to(device)
        lengths = (src != pad_idx).sum(dim=0)

        optimizer.zero_grad()
        output = model(src, lengths, tgt, teacher_forcing_ratio=TEACHER_FORCING_RATIO)
        loss = criterion(output.reshape(-1, output.size(-1)), tgt[1:].reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        avg_loss = total_loss / (idx + 1)

        tqdm_bar.set_postfix(loss=f"{avg_loss:.4f}")

    return total_loss / len(dataloader)

In [ ]:
def evaluation_step(dataloader, model, criterion, device, pad_idx):
    """perform an evaluation on one epoch

    Args:
        dataloader (DataLoader): the data loader for the evaluation data
        model (Seq2Seq): the sequence-to-sequence model
        criterion (torch.nn.Module): the loss function to use
        device (torch.device): the device to use for evaluation
        pad_idx (int): the index to use for padding

    Returns:
        float: the average loss for the epoch
    """
    model.eval()
    total_loss = 0

    tqdm_bar = tqdm(dataloader, desc="Evaluating", leave=False)
    with torch.no_grad():
        for idx, (src, tgt) in enumerate(tqdm_bar):
            src, tgt = src.to(device), tgt.to(device)
            lengths = (src != pad_idx).sum(dim=0)

            output = model(src, lengths, tgt, teacher_forcing_ratio=0.0)
            loss = criterion(output.reshape(-1, output.size(-1)), tgt[1:].reshape(-1))

            total_loss += loss.item()
            avg_loss = total_loss / (idx + 1)

            tqdm_bar.set_postfix(loss=f"{avg_loss:.4f}")

    return total_loss / len(dataloader)

In [ ]:
tokenizer = spm.SentencePieceProcessor(model_file=str(TOKENIZER_MODEL))

def encode_pairs(pairs):
    return [
        (
            tokenizer.encode(src),
            tokenizer.encode(tgt, add_bos=True, add_eos=True),
        )
        for src, tgt in pairs
    ]

train_pairs = encode_pairs(read_split(TRAIN_DATA))
valid_pairs = encode_pairs(read_split(VALID_DATA))

train_dataset = UQADataset(train_pairs)
valid_dataset = UQADataset(valid_pairs)

train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: collate_fn(x, PAD_IDX)
)
valid_dataloader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda x: collate_fn(x, PAD_IDX)
)

In [ ]:
model = Seq2Seq(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX,
    sos_idx=SOS_IDX,
    eos_idx=EOS_IDX,
    device=DEVICE
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_IDX)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

best_valid_loss = float("inf")
epoch_train_losses, epoch_valid_losses = [], []

In [ ]:
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"Epoch {epoch}")
    train_loss = train_step(train_dataloader, model, optimizer, criterion, DEVICE, PAD_IDX)
    valid_loss = evaluation_step(valid_dataloader, model, criterion, DEVICE, PAD_IDX)

    epoch_train_losses.append(train_loss)
    epoch_valid_losses.append(valid_loss)

    print(f"Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f}")

    torch.save(model.state_dict(), CHECKPOINT_DIR / f"epoch_{epoch}.pt")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), BEST_MODEL)
        print(f"Best model saved with loss: {best_valid_loss:.4f}")

In [ ]:
num_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / "model_summary.txt", "w", encoding="utf-8") as f:
    f.write(f"Number of trainable parameters: {num_parameters}\n")
    f.write(f"Epoch Train Losses: {epoch_train_losses}\n")
    f.write(f"Epoch Valid Losses: {epoch_valid_losses}\n")

In [ ]:
FIGS_DIR.mkdir(parents=True, exist_ok=True)
plt.figure()
plt.plot(range(1, NUM_EPOCHS + 1), epoch_train_losses, label="Train Loss")
plt.plot(range(1, NUM_EPOCHS + 1), epoch_valid_losses, label="Valid Loss")
plt.title("Training and Validation Losses")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig(FIGS_DIR / "loss_plot.png")